# 01 Station & Data Audit (Phase 1 Acceptance Criteria)

**Objective:** Evaluate OpenAQ station networks for Lahore (target domain) and Delhi (source domain) against the Phase 1 acceptance criteria and decision rules defined in `docs/data-engineering.md §13`.

### Decision Rules Evaluated:
1. **Lahore Station Eligibility & Sparsity:** $\ge 4$ eligible stations ($N_{max} = \text{eligible} - 3$). If $< 4$, widen target family to Lahore + Islamabad + Indian-Punjab.
2. **Regulatory Reference Monitor Anchor:** $\ge 1$ regulatory-grade monitor (`isMonitor`) required as truth anchor and calibration baseline.
3. **Winter-Day Completeness:** Fraction of winter smog-season days (Oct–Feb) with $\ge 18$ valid hours. If $< 50\%$, Phase 2 must adapt support window.
4. **Co-location Hierarchy:** Group stations $\le 50$ m with different providers; anchor to reference monitor if present, else median of corrected low-cost.
5. **Delhi Transfer Feasibility:** $\ge 20$ eligible stations across $\ge 5$ seasons; else enable Amritsar/Ludhiana source domains.

In [ ]:
import pandas as pd

from smogsense.analysis.station_audit import run_station_audit

print("SmogSense Station Audit Suite Loaded.")

## 1. Lahore Target Domain Audit

Evaluate target domain station eligibility, reference monitors, and co-located groups.

In [ ]:
lahore_summary, _ = run_station_audit(domain="lahore", output_dir=".state/audit")

print(f"Domain: {lahore_summary.domain.upper()}")
print(f"Total Stations Discovered: {lahore_summary.total_locations}")
print(
    f"Eligible Stations (90d uptime >= 50%, >= 30d history): {lahore_summary.eligible_station_count}"
)
print(f"Held-Out Sparsity Budget (N_max): {lahore_summary.n_max_held_out}")
print(f"Reference Monitors: {lahore_summary.reference_station_count}")
print(f"Winter Smog-Day Completeness: {lahore_summary.winter_day_completeness_pct}%")
print(f"Co-located Groups: {lahore_summary.colocated_groups_count}")

### Lahore Decision Rules Evaluation:

In [ ]:
for rule_id, rule in lahore_summary.decision_rules.items():
    print(f"[{rule['status']}] {rule_id}")
    print(f"   Threshold: {rule['threshold']}")
    print(f"   Observed:  {rule['observed']}")
    print(f"   Action:    {rule['recommendation']}\n")

## 2. Delhi Source Domain Audit

Evaluate source domain transfer learning feasibility ($\ge 20$ stations, $\ge 5$ seasons).

In [ ]:
delhi_summary, _ = run_station_audit(domain="delhi", output_dir=".state/audit")

print(f"Domain: {delhi_summary.domain.upper()}")
print(f"Total Stations Discovered: {delhi_summary.total_locations}")
print(f"Eligible Stations: {delhi_summary.eligible_station_count}")
print(f"Reference Monitors: {delhi_summary.reference_station_count}")

for rule_id, rule in delhi_summary.decision_rules.items():
    print(f"[{rule['status']}] {rule_id}: {rule['status']} -> {rule['recommendation']}")

## 3. Co-location Hierarchy & Truth Basis Summary

In [ ]:
if lahore_summary.colocated_groups:
    coloc_df = pd.DataFrame(lahore_summary.colocated_groups)
    print(coloc_df)
else:
    print("No co-located groups found in target network.")